In [13]:
import sqlite3
import pandas as pd
import numpy as np

# Points

In [14]:
conn = sqlite3.connect('nba_stats.db')

query = '''
SELECT b.slug, b.season_end_year, b.name, b.games_played, b.minutes_played, b.points, p.per_100_pts, a.pace
  FROM basic_stats AS b 
  JOIN per_100_stats as p
  ON b.slug = p.slug AND b.season_end_year = p.season_end_year
  JOIN nba_advanced_averages as a
  ON b.season_end_year = a.season_end_year
'''
possessions_df = pd.read_sql_query(query, conn)
conn.close()

pd.set_option('display.max_columns', None)
possessions_df.head()

,slug,season_end_year,name,games_played,minutes_played,points,per_100_pts,pace
0,greenac01,2001,A.C. Green,82,1411,367,13.8,92.584138
1,guytoaj01,2001,A.J. Guyton,33,630,198,16.9,92.584138
2,mckieaa01,2001,Aaron McKie,76,2394,878,18.9,92.584138
3,williaa01,2001,Aaron Williams,82,2336,838,18.5,92.584138
4,keefead01,2001,Adam Keefe,67,836,168,10.1,92.584138


In [15]:
possessions_df['possessions'] = (100 * possessions_df['points']) / possessions_df['per_100_pts']
#possessions_df['possessions'] = possessions_df['possessions'].round(0).astype(int) had an error with nulls
possessions_df.head()

,slug,season_end_year,name,games_played,minutes_played,points,per_100_pts,pace,possessions
0,greenac01,2001,A.C. Green,82,1411,367,13.8,92.584138,2659.420290
1,guytoaj01,2001,A.J. Guyton,33,630,198,16.9,92.584138,1171.597633
2,mckieaa01,2001,Aaron McKie,76,2394,878,18.9,92.584138,4645.502646
3,williaa01,2001,Aaron Williams,82,2336,838,18.5,92.584138,4529.729730
4,keefead01,2001,Adam Keefe,67,836,168,10.1,92.584138,1663.366337


In [16]:
possessions_nulls = possessions_df[possessions_df['possessions'].isnull()]
possessions_nulls

,slug,season_end_year,name,games_played,minutes_played,points,per_100_pts,pace,possessions
15,pankoan01,2001,Andy Panko,1,1,0,0.0,92.584138,NaN
29,longar01,2001,Art Long,9,20,0,0.0,92.584138,NaN
185,bowmair01,2001,Ira Bowman,3,19,0,0.0,92.584138,NaN
254,ketnela01,2001,Lari Ketner,3,7,0,0.0,92.584138,NaN
309,boguemu01,2001,Muggsy Bogues,3,34,0,0.0,92.584138,NaN
...,...,...,...,...,...,...,...,...,...
12219,pullizy01,2025,Zyon Pullin,3,3,0,0.0,99.579667,NaN
12316,hepbuch01,2026,Chucky Hepburn,2,13,0,0.0,100.216667,NaN
12344,brownda04,2026,Darius Brown II,1,3,0,0.0,100.216667,NaN
12654,essenno01,2026,Noa Essengue,2,6,0,0.0,100.216667,NaN


In [17]:
print(possessions_nulls.sort_values(by=['minutes_played'], ascending=True))

            slug  season_end_year                name  games_played  \
5488   jamesda01             2013        Damion James             2   
2225   scaleal01             2006         Alex Scales             1   
4208   curryja01             2010       JamesOn Curry             1   
10913  fostemi02             2023  Michael Foster Jr.             1   
3118   wafervo01             2007           Von Wafer             1   
...          ...              ...                 ...           ...   
309    boguemu01             2001       Muggsy Bogues             3   
1194   sanchpe01             2003        Pepe Sánchez             9   
7675   garinpa01             2017     Patricio Garino             5   
11565  arcidry01             2024    Ryan Arcidiacono            20   
3123   conrowi01             2007         Will Conroy             7   

       minutes_played  points  per_100_pts       pace  possessions  
5488                0       0          0.0  92.944000          NaN  
2225     

In [ ]:
possessions_no_nulls = possessions_df.copy()

estimated_possessions = (possessions_no_nulls['pace'] / 48) * possessions_no_nulls['minutes_played']
possessions_no_nulls['possessions'] = possessions_no_nulls['possessions'].fillna(estimated_possessions)
print(possessions_no_nulls[possessions_no_nulls['slug'] == 'longar01'])

         slug  season_end_year      name  games_played  minutes_played  \
29   longar01             2001  Art Long             9              20   
471  longar01             2002  Art Long            63             989   
911  longar01             2003  Art Long            26             211   

     points  per_100_pts       pace  possessions  
29        0          0.0  92.584138    38.576724  
471     285         15.3  91.957241  1862.745098  
911      60         13.8  92.300345   434.782609  


In [22]:
possessions_no_nulls['possessions'] = possessions_no_nulls['possessions'].round(0).astype(int)
possessions_no_nulls.head()

,slug,season_end_year,name,games_played,minutes_played,points,per_100_pts,pace,possessions
0,greenac01,2001,A.C. Green,82,1411,367,13.8,92.584138,2659
1,guytoaj01,2001,A.J. Guyton,33,630,198,16.9,92.584138,1172
2,mckieaa01,2001,Aaron McKie,76,2394,878,18.9,92.584138,4646
3,williaa01,2001,Aaron Williams,82,2336,838,18.5,92.584138,4530
4,keefead01,2001,Adam Keefe,67,836,168,10.1,92.584138,1663
